# LIBRAS to Portuguese Translation

**CNN + LSTM pipeline** for translating Brazilian Sign Language to Portuguese text using MediaPipe hand landmarks.

- **Architecture**: CNN Encoder → LSTM → Text Output
- **Dataset**: ~28,602 samples (original + augmented)
- **Input**: Hand landmark sequences (21 points × 3 coordinates)
- **Output**: Portuguese text tokens

In [40]:
# 📦 Import Libraries and Setup
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import datetime
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer
import gc

# 🔧 Device and Memory Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.8)
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU Memory: 4.22 GB


## 1. Dataset Loading

In [41]:
class SignLanguageDataset(Dataset):
    def __init__(self, csv_paths, landmarks_dirs=None):
        if isinstance(csv_paths, str):
            csv_paths = [csv_paths]
        
        dataframes = []
        for csv_path in csv_paths:
            df = pd.read_csv(csv_path)
            df = df.dropna(subset=['landmark_path', 'phrase'])
            dataframes.append(df)
        
        self.metadata = pd.concat(dataframes, ignore_index=True)
        self.landmarks_dirs = landmarks_dirs or []

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        landmark_filename = row['landmark_path']
        
        # Find landmark file
        landmark_path = None
        if landmark_filename.startswith('../'):
            landmark_path = landmark_filename
        else:
            for landmarks_dir in self.landmarks_dirs:
                potential_path = os.path.join(landmarks_dir, landmark_filename)
                if os.path.exists(potential_path):
                    landmark_path = potential_path
                    break
        
        if landmark_path is None or not os.path.exists(landmark_path):
            raise FileNotFoundError(f"Landmark file not found: {landmark_filename}")
        
        landmarks = np.load(landmark_path)
        if landmarks.shape[0] == 1:
            landmarks = landmarks.squeeze(0)
            
        phrase = row['phrase']
        return torch.tensor(landmarks, dtype=torch.float32), phrase

# 📁 Load Dataset
CSV_PATHS = [
    '../dataset/labels_metadata.csv',
    '../dataset/augmented_landmark_labels.csv'
]
LANDMARKS_DIRS = [
    '../dataset/processed/landmarks',
    '../dataset/processed/landmarks_augmented'
]

full_dataset = SignLanguageDataset(CSV_PATHS, LANDMARKS_DIRS)
print(f"Dataset loaded: {len(full_dataset)} samples")

Dataset loaded: 28602 samples


In [34]:
# 🔍 Analyze Data Dimensions
# Let's check the actual landmark structure
sample_landmarks, sample_phrase = full_dataset[0]
print(f"Sample landmark shape: {sample_landmarks.shape}")
print(f"Expected: [sequence_length, n_keypoints, 3]")
print(f"Sample phrase: {sample_phrase}")

# Check a few more samples to understand the data structure
for i in range(min(3, len(full_dataset))):
    landmarks, phrase = full_dataset[i]
    print(f"Sample {i}: shape={landmarks.shape}, phrase='{phrase}'")

# Determine the actual number of keypoints
if len(sample_landmarks.shape) == 3:
    actual_n_points = sample_landmarks.shape[1]
    print(f"\n🎯 Detected keypoints per frame: {actual_n_points}")
    if actual_n_points == 42:
        print("   📋 Data contains 42 points (2 hands × 21 points each)")
    elif actual_n_points == 21:
        print("   📋 Data contains 21 points (single hand)")
    else:
        print(f"   📋 Data contains {actual_n_points} points (custom configuration)")
else:
    print(f"⚠️ Unexpected data shape: {sample_landmarks.shape}")

Sample landmark shape: torch.Size([151, 42, 3])
Expected: [sequence_length, n_keypoints, 3]
Sample phrase: Vinte
Sample 0: shape=torch.Size([151, 42, 3]), phrase='Vinte'
Sample 1: shape=torch.Size([312, 42, 3]), phrase='Vinte'
Sample 2: shape=torch.Size([333, 42, 3]), phrase='Vinte'

🎯 Detected keypoints per frame: 42
   📋 Data contains 42 points (2 hands × 21 points each)


## 2. Tokenizer

In [42]:
# 🔤 Portuguese BERT Tokenizer
hf_tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

def encode_text(text):
    return hf_tokenizer.encode(text, add_special_tokens=False)

print(f"Vocabulary size: {hf_tokenizer.vocab_size}")

Vocabulary size: 29794


## 3. Data Preparation

In [43]:
# 📊 Dataset Split (80/10/10)
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1
total_size = len(full_dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

def collate_fn(batch):
    inputs, phrases = zip(*batch)
    inputs_list = [inp for inp in inputs]
    inputs_padded = pad_sequence(inputs_list, batch_first=True, padding_value=0.0)
    encoded = [torch.tensor(encode_text(phrase)) for phrase in phrases]
    targets = pad_sequence(encoded, batch_first=True, padding_value=0)
    return inputs_padded, targets

# 🔄 DataLoaders (Memory Optimized)
batch_size = 4
num_workers = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn,
                         num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn,
                       num_workers=num_workers, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn,
                        num_workers=num_workers, pin_memory=torch.cuda.is_available())

print(f"Dataset: Train={train_size}, Val={val_size}, Test={test_size}")
print(f"Batch size: {batch_size}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

Dataset: Train=22881, Val=2860, Test=2861
Batch size: 4


## 4. Model Architecture

In [44]:
class CNNEncoder(nn.Module):
    def __init__(self, n_points, emb_size):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=3, out_channels=32, kernel_size=1, padding=0)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(32, 64, kernel_size=1, padding=0)
        self.fc = nn.Linear(64 * n_points, emb_size)

    def forward(self, x):
        batch_size, seq_len, n_points, _ = x.shape
        x = x.view(-1, n_points, 3).permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(-1, 64 * n_points)
        x = self.fc(x)
        x = x.view(batch_size, seq_len, -1)
        return x

class Sign2TextModel(nn.Module):
    def __init__(self, n_points, emb_size, hidden_size, vocab_size, num_layers=1, dropout=0.3):
        super().__init__()
        self.encoder = CNNEncoder(n_points, emb_size)
        self.rnn = nn.LSTM(emb_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.encoder(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

## 5. Model Configuration

In [45]:
# ⚙️ Model Configuration
config = {
    'num_epochs': 100,
    'batch_size': 4,
    'learning_rate': 0.001,
    'dropout': 0.3,
    'emb_size': 128,
    'hidden_size': 256,
    'num_layers': 1,
    'n_points': 42,  # Two hands (42 keypoints total: 2 × 21)
    'vocab_size': hf_tokenizer.vocab_size,
    'resume_from_checkpoint': '../models/best_model_270625.pt',
    'save_with_date': True
}

def init_weights(m):
    if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

# 🏗️ Model Initialization
model = Sign2TextModel(
    config['n_points'], config['emb_size'], config['hidden_size'], 
    config['vocab_size'], config['num_layers'], config['dropout']
)

# 📂 Load Checkpoint (if available)
start_epoch = 0
checkpoint_path = config['resume_from_checkpoint']

if checkpoint_path and os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
                start_epoch = checkpoint.get('epoch', 0)
            else:
                model.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)
            
        print(f"Checkpoint loaded (epoch {start_epoch})")
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        model.apply(init_weights)
        print("Initialized with random weights")
else:
    model.apply(init_weights)
    print("Initialized with random weights")

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training starts from epoch: {start_epoch + 1}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading checkpoint: ../models/best_model_270625.pt
Error loading checkpoint: Error(s) in loading state_dict for Sign2TextModel:
	size mismatch for encoder.fc.weight: copying a param with shape torch.Size([128, 1344]) from checkpoint, the shape in current model is torch.Size([128, 2688]).
Initialized with random weights
Model parameters: 8,398,754
Training starts from epoch: 1


/home/alunotgn/Documentos/TalesOliveira/ifb_tcc/venv/lib/python3.10/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


## 6. Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs, start_epoch=0, save_with_date=True):
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    train_losses = []
    val_losses = []
    accumulation_steps = 2
    
    os.makedirs('../models', exist_ok=True)
    current_date = datetime.datetime.now().strftime("%d%m%y")
    
    print(f"Training: epoch {start_epoch + 1} → {num_epochs}")
    
    for epoch in range(start_epoch, num_epochs):
        # Training
        model.train()
        running_train_loss = 0.0
        train_batches = 0
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        for batch_idx, batch in enumerate(train_loader):
            inputs, targets = batch
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(inputs)
            batch_size, seq_len, vocab_size = outputs.shape
            outputs_flat = outputs.reshape(-1, vocab_size)
            targets_flat = targets.reshape(-1)
            
            mask = targets_flat != 0
            if mask.sum() == 0:
                continue
                
            outputs_valid = outputs_flat[mask.nonzero(as_tuple=True)[0]]
            targets_valid = targets_flat[mask]
            
            loss = criterion(outputs_valid, targets_valid.long()) / accumulation_steps
            loss.backward()
            
            if (batch_idx + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
                if torch.cuda.is_available() and (batch_idx + 1) % (accumulation_steps * 10) == 0:
                    torch.cuda.empty_cache()
            
            running_train_loss += loss.item() * accumulation_steps
            train_batches += 1

        if train_batches % accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        avg_train_loss = running_train_loss / train_batches if train_batches > 0 else 0
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for batch in val_loader:
                inputs, targets = batch
                inputs = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                
                outputs = model(inputs)
                batch_size, seq_len, vocab_size = outputs.shape
                outputs_flat = outputs.reshape(-1, vocab_size)
                targets_flat = targets.reshape(-1)
                mask = targets_flat != 0
                
                if mask.sum() == 0:
                    continue
                    
                outputs_valid = outputs_flat[mask.nonzero(as_tuple=True)[0]]
                targets_valid = targets_flat[mask]
                loss = criterion(outputs_valid, targets_valid.long())
                
                running_val_loss += loss.item()
                val_batches += 1
        
        avg_val_loss = running_val_loss / val_batches if val_batches > 0 else 0
        val_losses.append(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train: {avg_train_loss:.4f}, Val: {avg_val_loss:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            
            best_model_path = f'../models/model_{current_date}.pt' if save_with_date else '../models/model.pt'
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'config': config,
                'training_date': current_date
            }
            torch.save(checkpoint, best_model_path)
            print(f"  ✅ Best model saved: {best_model_path}")
        
        # Save periodic checkpoint
        if (epoch + 1) % 10 == 0:
            checkpoint_path = f'../models/checkpoint_epoch_{epoch+1}_{current_date}.pt' if save_with_date else f'../models/checkpoint_epoch_{epoch+1}.pt'
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'config': config,
                'training_date': current_date
            }
            torch.save(checkpoint, checkpoint_path)
            print(f"  💾 Checkpoint saved: {checkpoint_path}")
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
    print("Training completed")
    model.load_state_dict(best_model_wts)
    return model, train_losses, val_losses, current_date

## 7. Training

In [ ]:
# 🚀 Start Training
print("Starting model training...")
print(f"Samples: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
print(f"Resume from: {config['resume_from_checkpoint'] if config['resume_from_checkpoint'] and os.path.exists(config['resume_from_checkpoint']) else 'Fresh start'}")
print("-" * 50)

model, train_losses, val_losses, training_date = train_model(
    model, train_loader, val_loader, criterion, optimizer, device, 
    config['num_epochs'], start_epoch=start_epoch, save_with_date=config['save_with_date']
)

print("-" * 50)
print("✅ Training completed!")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")
print(f"Best model: ../models/best_model_{training_date}.pt")

## 8. Model Evaluation

In [ ]:
# 🎯 Comprehensive Model Evaluation

def evaluate_model(model, dataloader, device, dataset_name="", model_config=None):
    # Evaluate model accuracy on a dataset with optional data adaptation
    model.eval()
    total_tokens = 0
    correct_tokens = 0
    
    # Check if we need to adapt data dimensions
    adaptation_type = model_config.get('data_adaptation', None) if model_config else None
    
    with torch.no_grad():
        for batch in dataloader:
            inputs, targets = batch
            
            # Adapt input data if needed
            if adaptation_type == 'use_first_hand_only' and inputs.shape[2] == 42:
                # Use only first 21 keypoints (first hand)
                inputs = inputs[:, :, :21, :]
            elif adaptation_type == 'pad_to_42_keypoints' and inputs.shape[2] == 21:
                # Pad from 21 to 42 keypoints by duplicating the hand data
                inputs = torch.cat([inputs, inputs], dim=2)  # Duplicate hand to simulate 2 hands
            
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            
            # Ensure we only compare valid sequence lengths
            # outputs shape: [batch_size, seq_len, vocab_size]
            # targets shape: [batch_size, target_seq_len]
            
            batch_size = outputs.shape[0]
            output_seq_len = outputs.shape[1]
            target_seq_len = targets.shape[1]
            
            # Take minimum sequence length to avoid dimension mismatch
            min_seq_len = min(output_seq_len, target_seq_len)
            
            # Truncate both tensors to the same length
            outputs_truncated = outputs[:, :min_seq_len, :]
            targets_truncated = targets[:, :min_seq_len]
            
            predicted = outputs_truncated.argmax(dim=-1)
            
            mask = (targets_truncated != 0)
            correct = (predicted == targets_truncated) & mask
            correct_tokens += correct.sum().item()
            total_tokens += mask.sum().item()
    
    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0
    
    # Create adaptation note for display
    adaptation_note = ""
    if adaptation_type == 'use_first_hand_only':
        adaptation_note = " (1st hand only)"
    elif adaptation_type == 'pad_to_42_keypoints':
        adaptation_note = " (padded to 42)"
    elif adaptation_type == 'dimension_mismatch':
        adaptation_note = " (dimension mismatch)"
    
    print(f"{dataset_name}{adaptation_note}: {accuracy * 100:.2f}%")
    return accuracy

def load_model_with_config(model_path, device):
    # Load model with automatic dimension detection
    try:
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # Detect data dimensions from current dataset
        sample_data, _ = full_dataset[0]
        actual_n_points = sample_data.shape[1] if len(sample_data.shape) == 3 else 21
        
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # Detect model dimensions from checkpoint
            state_dict = checkpoint['model_state_dict']
            fc_weight_shape = state_dict['encoder.fc.weight'].shape
            checkpoint_emb_size = fc_weight_shape[0]  # Output dimension
            checkpoint_input_size = fc_weight_shape[1]  # Input dimension (64 * n_points)
            checkpoint_n_points = checkpoint_input_size // 64  # Assuming 64 channels from conv
            
            print(f"🔍 Model was trained with: {checkpoint_n_points} keypoints")
            print(f"🔍 Current data has: {actual_n_points} keypoints")
            
            # Always create model with checkpoint dimensions, then adapt data if needed
            if 'config' in checkpoint:
                cfg = checkpoint['config'].copy()
                # Override n_points with detected value from checkpoint
                cfg['n_points'] = checkpoint_n_points
            else:
                cfg = {
                    'n_points': checkpoint_n_points,
                    'emb_size': checkpoint_emb_size,
                    'hidden_size': 256,
                    'vocab_size': hf_tokenizer.vocab_size,
                    'num_layers': 1,
                    'dropout': 0.3
                }
            
            # Create model with checkpoint dimensions
            eval_model = Sign2TextModel(
                cfg['n_points'], cfg['emb_size'], cfg['hidden_size'], 
                cfg['vocab_size'], cfg['num_layers'], cfg['dropout']
            ).to(device)
            
            eval_model.load_state_dict(checkpoint['model_state_dict'])
            
            # Set data adaptation if needed
            if checkpoint_n_points != actual_n_points:
                if checkpoint_n_points == 21 and actual_n_points == 42:
                    print("🔄 Will use only first hand (21 keypoints) to match model")
                    cfg['data_adaptation'] = 'use_first_hand_only'
                elif checkpoint_n_points == 42 and actual_n_points == 21:
                    print("⚠️ Model expects 42 keypoints but data has 21 - this may not work well")
                    cfg['data_adaptation'] = 'pad_to_42_keypoints'
                else:
                    print(f"⚠️ Unusual dimension mismatch: {checkpoint_n_points} → {actual_n_points}")
                    cfg['data_adaptation'] = 'dimension_mismatch'
            else:
                print("✅ Dimensions match perfectly")
            
            return eval_model, cfg
        
        else:
            # Legacy model format - need to detect dimensions from state dict
            print("🔄 Loading legacy model format")
            
            # Try to detect dimensions from the legacy checkpoint
            if 'encoder.fc.weight' in checkpoint:
                fc_weight_shape = checkpoint['encoder.fc.weight'].shape
                checkpoint_emb_size = fc_weight_shape[0]
                checkpoint_input_size = fc_weight_shape[1]
                checkpoint_n_points = checkpoint_input_size // 64
                
                print(f"🔍 Legacy model was trained with: {checkpoint_n_points} keypoints")
                
                cfg = {
                    'n_points': checkpoint_n_points,
                    'emb_size': checkpoint_emb_size,
                    'hidden_size': 256,
                    'vocab_size': hf_tokenizer.vocab_size,
                    'num_layers': 1,
                    'dropout': 0.3
                }
                
                # Set data adaptation if needed
                if checkpoint_n_points != actual_n_points:
                    if checkpoint_n_points == 21 and actual_n_points == 42:
                        print("🔄 Will use only first hand (21 keypoints) to match legacy model")
                        cfg['data_adaptation'] = 'use_first_hand_only'
                    else:
                        print(f"⚠️ Legacy model dimension mismatch: {checkpoint_n_points} → {actual_n_points}")
                        cfg['data_adaptation'] = 'dimension_mismatch'
                
            else:
                # Fallback to current data dimensions
                cfg = {
                    'n_points': actual_n_points,
                    'emb_size': 128,
                    'hidden_size': 256,
                    'vocab_size': hf_tokenizer.vocab_size,
                    'num_layers': 1,
                    'dropout': 0.3
                }
            
            eval_model = Sign2TextModel(
                cfg['n_points'], cfg['emb_size'], cfg['hidden_size'], 
                cfg['vocab_size'], cfg['num_layers'], cfg['dropout']
            ).to(device)
            
            eval_model.load_state_dict(checkpoint)
            return eval_model, cfg
            
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None

# 🔧 Select and Load Model
selected_model_name = "model_270625.pt"  # Change this to evaluate different models
models_dir = '../models'
selected_model_path = os.path.join(models_dir, selected_model_name)

print(f"📋 Selected Model: {selected_model_name}")
print(f"🔍 Data Dimensions: {full_dataset[0][0].shape[1]} keypoints per frame")

if os.path.exists(selected_model_path):
    file_size = os.path.getsize(selected_model_path) / (1024 * 1024)
    print(f"✅ Model found ({file_size:.1f} MB)")
    
    # Load model metadata
    try:
        checkpoint = torch.load(selected_model_path, map_location='cpu')
        if isinstance(checkpoint, dict):
            if 'epoch' in checkpoint:
                print(f"📊 Training epochs: {checkpoint['epoch']}")
            if 'best_val_loss' in checkpoint:
                print(f"📉 Best validation loss: {checkpoint['best_val_loss']:.4f}")
            if 'training_date' in checkpoint:
                print(f"📅 Training date: {checkpoint['training_date']}")
    except Exception as e:
        print(f"⚠️ Could not load metadata: {e}")
    
    # Load and evaluate model
    print("\n🔄 Loading model for evaluation...")
    eval_model, model_config = load_model_with_config(selected_model_path, device)
    
    if eval_model is not None:
        print("✅ Model loaded successfully!")
        print(f"🏗️ Architecture: {model_config['n_points']} keypoints → {model_config['emb_size']} embedding → {model_config['hidden_size']} LSTM → {model_config['vocab_size']} vocab")
        
        # Run comprehensive evaluation
        print("\n📊 Evaluation Results:")
        print("-" * 40)
        
        results = {}
        results['train'] = evaluate_model(eval_model, train_loader, device, "Training Set  ", model_config)
        results['val'] = evaluate_model(eval_model, val_loader, device, "Validation Set", model_config)
        results['test'] = evaluate_model(eval_model, test_loader, device, "Test Set     ", model_config)
        
        # Save detailed results
        import json
        results_file = f"../models/eval_{selected_model_name.replace('.pt', '')}_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"
        
        detailed_results = {
            'model_name': selected_model_name,
            'model_config': model_config,
            'data_dimensions': {
                'keypoints_per_frame': full_dataset[0][0].shape[1],
                'sequence_length_example': full_dataset[0][0].shape[0],
                'coordinate_dimensions': full_dataset[0][0].shape[2]
            },
            'accuracies': results,
            'evaluation_date': datetime.datetime.now().isoformat(),
            'dataset_sizes': {
                'train': len(train_dataset),
                'validation': len(val_dataset),
                'test': len(test_dataset)
            }
        }
        
        with open(results_file, 'w') as f:
            json.dump(detailed_results, f, indent=2)
        
        print(f"\n💾 Detailed results saved: {results_file}")
        print("-" * 40)
        
        # Memory cleanup
        del eval_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    else:
        print("❌ Failed to load model!")
        
else:
    print("❌ Model not found!")
    try:
        available_models = [f for f in os.listdir(models_dir) if f.endswith('.pt')]
        print(f"📁 Available models: {available_models}")
    except Exception as e:
        print(f"Could not list models directory: {e}")

📋 Selected Model: best_model_270625.pt
🔍 Data Dimensions: 42 keypoints per frame
✅ Model found (31.4 MB)

🔄 Loading model for evaluation...
🔄 Loading legacy model format
🔍 Legacy model was trained with: 21 keypoints
🔄 Will use only first hand (21 keypoints) to match legacy model
✅ Model loaded successfully!
🏗️ Architecture: 21 keypoints → 128 embedding → 256 LSTM → 29794 vocab

📊 Evaluation Results:
----------------------------------------
Training Set   (1st hand only): 0.10%
Training Set   (1st hand only): 0.10%
Validation Set (1st hand only): 0.11%
Validation Set (1st hand only): 0.11%
Test Set      (1st hand only): 0.20%

💾 Detailed results saved: ../models/eval_best_model_270625_20250722_1034.json
----------------------------------------
Test Set      (1st hand only): 0.20%

💾 Detailed results saved: ../models/eval_best_model_270625_20250722_1034.json
----------------------------------------


In [ ]:
# 🧹 Final Memory Cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ GPU memory cleaned")